# 05 · Temporal kNN on the d8 residual — relevance / analog probes

*Edge-features arc · 01 discovery · 02 linear base · 03 hero cascade · 04 regime-MoE · 05 temporal-kNN  —  machinery: `ridge_pipeline_throughline.ipynb`*

**Verify first — nothing pasted.** A model-free probe of the *same* d8@cs0.5 leftover the MoE/EBM model: predict the
h16-19 residual from **regime-similar past close bars** (self-attention ≈ learned kNN; here a fixed-metric causal
kNN). The Hero-A base QLIKE below is **recomputed from the saved leftover preds** (`preds/fa_d8c5.csv`) with the
real `apply_duan_smearing` metric and asserted to 0.12081; the two leakage-guarded variants (`knn_d8.sbatch`) are
read from `knn_d8.csv`, and each carries a **shuffle placebo** (shuffle the neighbour residuals -> must give ~0;
a gain there = look-ahead artifact) which the Verify cell asserts is `<5e-4`.

In [1]:
import html, inspect, json, os, sys, textwrap
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

def find_repo(s):
    for q in [Path(s).resolve(), *Path(s).resolve().parents]:
        if (q / "resid_amortized.py").exists() and (q / "src").is_dir():
            return q
    raise FileNotFoundError("repo root")
REPO = find_repo(Path.cwd()); os.chdir(REPO); sys.path.insert(0, str(REPO))
LADDER = REPO / "results" / "moe_ladder"

# Collapsible, theme-following source display (one <details> per object; folded).
def _details(f, open_=False):
    mod = f.__module__.replace("src.", "src/").replace(".", "/") + ".py"
    try:
        sig = ("class " + f.__name__) if inspect.isclass(f) else ("def " + f.__name__ + str(inspect.signature(f)))
    except (ValueError, TypeError):
        sig = f.__qualname__
    body = "```python\n" + textwrap.dedent(inspect.getsource(f)).rstrip() + "\n```"
    return (f"<details{' open' if open_ else ''}>\n<summary><code>{html.escape(mod + '  ·  ' + sig)}"
            f"</code></summary>\n\n{body}\n\n</details>")
def show_one(f):
    return Markdown(_details(f))
def show_src(path, lang="python", open_=False):  # fold a whole source FILE (scripts have no importable fn)
    p = Path(path); txt = p.read_text(encoding="utf-8").rstrip()
    return Markdown(f"<details{' open' if open_ else ''}>\n<summary><code>{html.escape(str(p))}"
                    f"</code></summary>\n\n```{lang}\n{txt}\n```\n\n</details>")

from src.evaluation.metrics import apply_duan_smearing  # the real metric (folded in s1; recomputed, never pasted)

def qlike(pred_adj, y_true, base):
    """Exact QLIKE the kNN scripts report: Duan-smear to raw scale, then mean(r - log r - 1)."""
    pr, tr = apply_duan_smearing(np.asarray(pred_adj, np.float64),
                                 np.asarray(y_true, np.float64), np.asarray(base, np.float64))
    m = (tr > 0) & (pr > 0); r = tr[m] / pr[m]
    return float(np.mean(r - np.log(r) - 1.0))
print("setup ok")

setup ok


---
## 1 . The two variants + the metric (folded)

- **`knn_analog`** -- causal kNN analog: weighted **mean** of the K nearest past close analogs, strict
  embargo >= HAR max lag (3125 bars) so neighbour residuals are fully realised. Carries a shuffle placebo.
- **`knn_local`** -- large-K **local-linear ridge** relevance regression (the "proper" variant; naive
  K-mean over-adds variance and hurts). Prior standalone: a sliver on the close leftover, shuffle-clean,
  but **dominated by the EBM regime**. Re-run here on the linbest `fa_d8c5` base for an apples-to-apples row.

The **leakage guards** live in the folded source: the embargo loop (`thr = kc[i] - EMBARGO`, `EMBARGO = 3125`)
and the SHUFFLE placebo (`rng.permutation` over the neighbour residuals). `apply_duan_smearing` is the metric
the Verify cell recomputes the base QLIKE with.

In [2]:
import knn_analog, knn_local
display(show_one(knn_analog.main))      # causal kNN analog -- embargo>=3125 + shuffle placebo
display(show_one(knn_local.main))       # large-K local-linear ridge relevance + shuffle placebo
display(show_one(apply_duan_smearing))  # the metric the Verify cell recomputes with (no pasted numbers)

<details>
<summary><code>knn_analog.py  ·  def main() -&gt; None</code></summary>

```python
def main() -> None:
    d = f"{CACHE_ROOT}/{CELL}"
    feats = json.load(open(f"{d}/feats.json"))
    Xs = np.load(f"{d}/Xs.npy", mmap_mode="r")

    parts = [
        pd.read_csv(f) for f in glob.glob(f"results/resid_ab/{CELL}/{LBL}/chunk_*.csv")
    ]
    A = pd.concat(parts, ignore_index=True).sort_values("k")
    k = A["k"].to_numpy().astype(np.int64)
    pred_adj = A["pred_adj"].to_numpy(np.float64)
    y_true = A["y_true"].to_numpy(np.float64)
    base = A["base"].to_numpy(np.float64)
    t = k + TRAIN_WIN

    hour = np.asarray(Xs[t, feats.index("hour")], dtype=np.float64)
    E = np.column_stack(
        [np.asarray(Xs[t, feats.index(c)], dtype=np.float64) for c in EMB_COLS] + [hour]
    )
    E = (E - E.mean(0)) / (E.std(0) + 1e-9)
    r = y_true - pred_adj
    close = (hour >= 16) & (hour <= 19)
    ci = np.where(close)[0]
    kc, Ec, rc = k[ci], np.ascontiguousarray(E[ci]), r[ci]
    nC = len(ci)
    print(
        "OOS rows=%d close bars=%d  embargo=%d  emb_dim=%d"
        % (len(k), nC, EMBARGO, E.shape[1]),
        flush=True,
    )

    def anhat(rsrc, K):
        ah = np.zeros(nC, dtype=np.float64)
        ptr = 0  # eligible = close bars [0:ptr) with kc <= kc[i]-EMBARGO (kc ascending)
        for i in range(nC):
            thr = kc[i] - EMBARGO
            while ptr < nC and kc[ptr] <= thr:
                ptr += 1
            if ptr < K:
                continue
            dd = Ec[:ptr] - Ec[i]
            dist = np.einsum("ij,ij->i", dd, dd)
            nn = np.argpartition(dist, K)[:K]
            ah[i] = rsrc[nn].mean()
        return ah

    q0 = qlike(pred_adj, y_true, base)
    q0c = qlike(pred_adj[ci], y_true[ci], base[ci])
    print("Hero A: full=%.5f  h16-19=%.5f" % (q0, q0c), flush=True)
    rng = np.random.RandomState(0)
    for K in (25, 50, 100):
        ah = anhat(rc, K)
        new = pred_adj.copy()
        new[ci] += ah
        q1 = qlike(new, y_true, base)
        q1c = qlike(new[ci], y_true[ci], base[ci])
        ahs = anhat(rc[rng.permutation(nC)], K)
        news = pred_adj.copy()
        news[ci] += ahs
        qs = qlike(news, y_true, base)
        print(
            "K=%3d full %.5f (d%+.6f) | SHUFFLE %.5f (d%+.6f) | h16-19 %.5f (d%+.6f)  ah|mean|=%.2e"
            % (K, q1, q1 - q0, qs, qs - q0, q1c, q1c - q0c, np.abs(ah[ah != 0]).mean()),
            flush=True,
        )
    print("KNN_ANALOG_DONE", flush=True)
```

</details>

<details>
<summary><code>knn_local.py  ·  def main() -&gt; None</code></summary>

```python
def main() -> None:
    d = f"{CACHE_ROOT}/{CELL}"
    feats = json.load(open(f"{d}/feats.json"))
    Xs = np.load(f"{d}/Xs.npy", mmap_mode="r")
    A = pd.concat(
        [
            pd.read_csv(f)
            for f in glob.glob(f"results/resid_ab/{CELL}/{LBL}/chunk_*.csv")
        ]
    ).sort_values("k")
    k = A["k"].to_numpy().astype(np.int64)
    pred_adj, y_true, base = (
        A[c].to_numpy(np.float64) for c in ("pred_adj", "y_true", "base")
    )
    r = y_true - pred_adj
    t = k + TRAIN_WIN
    hour = np.asarray(Xs[t, feats.index("hour")], dtype=np.float64)
    close = (hour >= 16) & (hour <= 19)
    ci = np.where(close)[0]
    kc, rc = k[ci], r[ci]
    tc = t[ci]

    def grab(cols):
        return (
            np.ascontiguousarray(
                np.column_stack(
                    [np.asarray(Xs[tc, feats.index(c)], dtype=np.float64) for c in cols]
                )
            )
            if cols
            else np.zeros((len(ci), 0))
        )

    Esim = grab(SIM)
    Esim = (Esim - Esim.mean(0)) / (Esim.std(0) + 1e-9)
    Fmats = {name: grab(cols) for name, cols in FSETS.items()}
    nC = len(ci)
    print(
        "close=%d  K=%d  embargo=%d  resid_std=%.4f" % (nC, K, EMBARGO, rc.std()),
        flush=True,
    )

    preds = {name: np.zeros(nC) for name in FSETS}
    rng = np.random.RandomState(0)
    rc_shuf = rc[rng.permutation(nC)]
    preds_shuf_all = np.zeros(nC)  # 'all' fset on shuffled residuals (placebo)
    ptr = 0
    for i in range(nC):
        thr = kc[i] - EMBARGO
        while ptr < nC and kc[ptr] <= thr:
            ptr += 1
        if ptr <= K:
            continue
        dd = Esim[:ptr] - Esim[i]
        dist = np.einsum("ij,ij->i", dd, dd)
        nn = np.argpartition(dist, K)[:K]
        for name, F in Fmats.items():
            Dtr = np.column_stack([np.ones(K), F[nn]])
            G = Dtr.T @ Dtr + RIDGE * np.eye(Dtr.shape[1])
            beta = np.linalg.solve(G, Dtr.T @ rc[nn])
            preds[name][i] = np.concatenate([[1.0], F[i]]) @ beta
        Da = np.column_stack([np.ones(K), Fmats["all"][nn]])
        Ga = Da.T @ Da + RIDGE * np.eye(Da.shape[1])
        bshuf = np.linalg.solve(Ga, Da.T @ rc_shuf[nn])
        preds_shuf_all[i] = np.concatenate([[1.0], Fmats["all"][i]]) @ bshuf

    q0 = qlike(pred_adj, y_true, base)
    q0c = qlike(pred_adj[ci], y_true[ci], base[ci])
    print("Hero A: full=%.5f  h16-19=%.5f" % (q0, q0c), flush=True)
    for name in FSETS:
        new = pred_adj.copy()
        new[ci] += preds[name]
        print(
            "  %-9s full %.5f (d%+.6f)  h16-19 %.5f (d%+.6f)"
            % (
                name,
                qlike(new, y_true, base),
                qlike(new, y_true, base) - q0,
                qlike(new[ci], y_true[ci], base[ci]),
                qlike(new[ci], y_true[ci], base[ci]) - q0c,
            ),
            flush=True,
        )
    new = pred_adj.copy()
    new[ci] += preds_shuf_all
    print(
        "  %-9s full %.5f (d%+.6f)  [should be ~0]"
        % ("SHUFFLE", qlike(new, y_true, base), qlike(new, y_true, base) - q0),
        flush=True,
    )
    print("KNN_LOCAL_DONE", flush=True)
```

</details>

<details>
<summary><code>src/evaluation/metrics.py  ·  def apply_duan_smearing(forecasts: &#x27;np.ndarray&#x27;, y_true: &#x27;np.ndarray&#x27;, baselines: &#x27;np.ndarray&#x27;) -&gt; &#x27;tuple[np.ndarray, np.ndarray]&#x27;</code></summary>

```python
def apply_duan_smearing(
    forecasts: np.ndarray,
    y_true: np.ndarray,
    baselines: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    """Apply Duan smearing correction to convert adjusted-scale forecasts to raw scale.

    Parameters
    ----------
    forecasts : array-like
        Model predictions on adjusted (sqrt / log) scale.
    y_true : array-like
        True values on adjusted scale.
    baselines : array-like
        Baseline volatility used to scale back to raw units.

    Returns
    -------
    pred_raw : np.ndarray
        Smearing-corrected predictions on raw scale.
    true_raw : np.ndarray
        True values on raw scale.
    """
    forecasts = np.asarray(forecasts, dtype=np.float64)
    y_true = np.asarray(y_true, dtype=np.float64)
    baselines = np.asarray(baselines, dtype=np.float64)

    smear = np.mean((y_true - forecasts) ** 2)
    pred_raw = (forecasts**2 + smear) * baselines
    true_raw = (y_true**2) * baselines
    return pred_raw, true_raw
```

</details>

---
## 2 . Verify -- recompute the Hero-A base, then the kNN deltas

(a) The Hero-A base QLIKE is **recomputed** from the saved leftover preds `preds/fa_d8c5.csv`
(cols `k, pred_adj, y_true, base`) via `qlike(...)` above -- asserted to reproduce **0.12081**.
(b) The kNN deltas are read from `knn_d8.csv` (parsed from the `knn_d8.sbatch` run): the naive analog
should **hurt** (variance), the proper local-ridge should **help a sliver**, and the **SHUFFLE** placebo
`|d_full|` must be `<5e-4` (leakage-clean). Reproduce-cmd: `bash knn_d8.sbatch`.

In [3]:
# (a) RECOMPUTE the Hero-A base QLIKE from the saved leftover preds (not pasted).
fp = LADDER / "preds" / "fa_d8c5.csv"
if fp.exists() and fp.stat().st_size > 0:
    fa = pd.read_csv(fp)  # cols: k, pred_adj, y_true, base
    q_base = qlike(fa.pred_adj, fa.y_true, fa.base)
    print(f"Hero-A base (fa_d8c5) recomputed QLIKE = {q_base:.5f}  (n={len(fa)})")
    assert round(q_base, 5) == 0.12081, f"Hero-A base did not reproduce 0.12081 (got {q_base:.5f})"
    print("PASS - Hero-A base reproduces 0.12081 from saved preds.")
else:
    print("PENDING - preds/fa_d8c5.csv absent. Reproduce: bash knn_d8.sbatch "
          "(writes the fa_d8c5 leftover chunks, collected here).")

# (b) The two kNN variants + the SHUFFLE placebo (parsed from the knn_d8 run).
kp = LADDER / "knn_d8.csv"
if kp.exists() and kp.stat().st_size > 0:
    knn = pd.read_csv(kp); display(knn)
    base_row = knn[knn.variant == "hero_a"]
    assert round(float(base_row.qlike_full.iloc[0]), 5) == 0.12081, "table base disagrees with the recompute"
    shuf = knn[knn.fset.astype(str).str.contains("SHUFFLE", case=False, na=False)]
    assert len(shuf) and (shuf.d_full.abs() < 5e-4).all(), "SHUFFLE placebo not ~0 -- look-ahead artifact!"
    analog = knn[knn.variant == "knn_analog"]
    local = knn[(knn.variant == "knn_local") & ~knn.fset.astype(str).str.contains("SHUFFLE", case=False, na=False)]
    assert (analog.d_full > 0).all(), "naive analog should HURT (variance) on this low-dim leftover"
    assert local.d_full.min() < 0, "proper local-ridge should help a sliver"
    print(f"PASS - SHUFFLE |d_full|={shuf.d_full.abs().max():.6f} < 5e-4 (leakage-clean); "
          f"analog hurts (+{analog.d_full.max():.5f}), local-ridge best d_full={local.d_full.min():+.5f}.")
else:
    print("PENDING - knn_d8.csv absent. Reproduce: bash knn_d8.sbatch "
          "(or: KNN_CELL=... KNN_LBL=resid_subset_fa_d8c5 $PY knn_analog.py ; $PY knn_local.py).")

Hero-A base (fa_d8c5) recomputed QLIKE = 0.12081  (n=194934)
PASS - Hero-A base reproduces 0.12081 from saved preds.


,variant,fset,qlike_full,d_full,qlike_h16_19,d_h16_19
0,hero_a,base,0.12081,0.000000,0.16070,0.000000
1,knn_analog,K25,0.12149,0.000673,0.16390,0.003198
2,knn_local,intercept,0.12079,-0.000030,0.16045,-0.000244
3,knn_local,har,0.12072,-0.000093,0.15986,-0.000837
4,knn_local,V,0.12071,-0.000108,0.15988,-0.000816
5,knn_local,all,0.12072,-0.000100,0.15982,-0.000875
6,knn_local,SHUFFLE,0.12109,0.000277,NaN,NaN


PASS - SHUFFLE |d_full|=0.000277 < 5e-4 (leakage-clean); analog hurts (+0.00067), local-ridge best d_full=-0.00011.


## 3 . Interpret

The expected story (now downstream of the verified deltas -- the table above), and why it matters even if it loses:

- **The proper local-linear kNN works** (placebo-clean, best `d_full` slightly negative) but is **dominated by the
  EBM regime** -- kNN is a *validated method, not a deployable lever here*. The naive K-mean analog **hurts**
  (positive `d_full`), which is itself a finding: the close leftover is low-dimensional and noise-dominated, so
  adding unstructured local variance is harmful -- only *structured* local fitting (ridge) extracts the thin signal.
- **kNN vs the MoE gate (ch. 04):** the MoE's soft gate is a *learned-metric* generalization of fixed kNN
  (the gate learns the regime the kNN's hand-metric assumes). If neither beats the EBM, the state framing
  is earned for real and the lever is **information (the auction cross), not more local/relevance modelling**.
- The model class that subsumes both (sequence-attention) is **provably empty here** (log-sig path-lever
  death, ch. 03) -- so the kNN is the right, data-efficient, *legible* probe, not a stepping stone to a
  sequence model.